In [ ]:
import csv
import glob
import os
import random
from decimal import Decimal
from IPython.display import clear_output

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [ ]:
seed = 42

In [ ]:
# 99以下＋９９以下＝１９８以下、の足し算であるが、
# 偽物ロジックの最大が2文字＋2文字＝4文字なので、
# 一行あたりのIDの数を１２に設定しておく
padding_length = 12


In [ ]:
# 登場する文字数が少ないので、自作の辞書を作成して使用する

chars = ["<pad>", "\n", "+", "=", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]

dict_char_to_id = {char: i for i, char in enumerate(chars)}
dict_id_to_char = {i: char for i, char in enumerate(chars)}

## 1. トイデータ作成

In [ ]:
sample_num = 10000 # サンプルデータを何個作るか
lower_limit = 0 # 乱数生成の下限
upper_limit = 99 # 乱数生成の上限

In [ ]:
# 本物のロジックでの計算式のサンプルデータ作成

list_true_logic = []

random.seed(seed)
for i in range(sample_num):

    x = random.randint(lower_limit, upper_limit)
    y = random.randint(lower_limit, upper_limit)

    sample_data = f"{x}+{y}={x+y}"

    list_true_logic.append(sample_data)

# データを保存
with open("./data/train/true_logic.txt", "w", encoding="utf-8", newline="") as f:
    f.write("\n".join(list_true_logic))

    

In [ ]:
# 偽物のロジックでの計算式のサンプルデータ作成

list_fake_logic = []

random.seed(seed)
for i in range(sample_num):

    x = random.randint(lower_limit, upper_limit)
    y = random.randint(lower_limit, upper_limit)

    sample_data = f"{x}+{y}={x}{y}"

    list_fake_logic.append(sample_data)

# データを保存
with open("./data/train/fake_logic.txt", "w", encoding="utf-8", newline="") as f:
    f.write("\n".join(list_fake_logic))

In [ ]:
# 謎ロジックというか、ランダムでのサンプルデータ作成

list_nonsense_logic = []

random.seed(seed)
for i in range(sample_num):

    x = random.randint(lower_limit, upper_limit)
    y = random.randint(lower_limit, upper_limit)

    # 左辺(x+y)が関係ない右辺を作成
    z = random.randint(lower_limit, upper_limit) + random.randint(lower_limit, upper_limit)

    sample_data = f"{x}+{y}={z}"

    list_nonsense_logic.append(sample_data)


# データを保存
with open("./data/train/nonsense_logic.txt", "w", encoding="utf-8", newline="") as f:
    f.write("\n".join(list_nonsense_logic))

## 2. トークナイズとID化

## 2.1 データの読み込み

In [ ]:
list_train_data_path = glob.glob("./data/train/*.txt")

In [ ]:
dfs = []

for file_path in list_train_data_path:
    df = pd.read_csv(file_path, names=[file_path.split(".")[1].split("/")[3]])

    dfs.append(df)

df_data = pd.concat(dfs, axis=1)

In [ ]:
df_data

### 2.2 トークナイズ

In [ ]:
df_toknized = df_data.copy()

In [ ]:
# 1文字ずつに分割
df_toknized = df_toknized.map(list)

In [ ]:
df_toknized.head(3)

In [ ]:
# 計算式の区切りの目印として、末尾に改行コードを挿入
df_toknized = df_toknized.map(lambda text_list: text_list + ["\n"])

In [ ]:
df_toknized.head(3)

In [ ]:
# パディング

df_toknized = df_toknized.apply(lambda col: col.apply(lambda x: x + ["<pad>"] * (padding_length - len(x))))

In [ ]:
df_toknized.head(3)

### 2.3 ID化

In [ ]:
# 文字をIDに変換

df_ids = df_toknized.map(lambda x: [dict_char_to_id.get(item, item) for item in x] if isinstance(x, list) else x).copy()

In [ ]:
df_ids.head(3)

## 3. データセット作成

In [ ]:
def make_dataset(df, true_ratio, fake_ratio, nonsense_ratio):
    
    if (true_ratio + fake_ratio + nonsense_ratio) != 1:
        raise ValueError("割合は合計が１になるようにしてください")

    random.seed(seed)
    list_return = []

    total_rows = len(df)

    list_return.extend(
        df["true_logic"].sample(n=int((total_rows * true_ratio).to_integral_value())).to_list()
    )
    list_return.extend(
        df["fake_logic"].sample(n=int((total_rows * fake_ratio).to_integral_value())).to_list()
    )
    list_return.extend(
        df["nonsense_logic"].sample(n=int((total_rows * nonsense_ratio).to_integral_value())).to_list()
    )

    # 本物計算ロジック、偽物計算ロジック、異次元ロジックの順で固まってるので、
    # データの偏りをなくす
    random.shuffle(list_return)
    
    # return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)
    # return list_return
    return torch.tensor(list_return, dtype=torch.long)
    

In [ ]:
# float型の内部表現対策でDecimalを使用
# ここまで厳密でなくともよいが、足して1にする。
true_ratio = Decimal("0.6")
fake_ratio = Decimal("0.1")
nonsense_ratio = Decimal("0.3")

# true_ratio = 0.7
# fake_ratio = 0.3
# nonsense_ratio = 0

datasets = make_dataset(
    df=df_ids,
    true_ratio=true_ratio,
    fake_ratio=fake_ratio,
    nonsense_ratio=nonsense_ratio
)

In [ ]:
# データセットができているか確認
datasets

## 4. モデル構築、学習

### 4.1 モデルの設定

In [ ]:
# シンプルな構成

class SimpleDecoderModel(nn.Module):
    
    # nheadがマルチヘッドアテンションのヘッドの数
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        
        super().__init__()

        # embedding
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # self-attentionとfeedforward layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=128,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.position_embedding = nn.Embedding(padding_length, d_model)

        self.fc_out = nn.Linear(d_model, vocab_size)
  
    def forward(self, x):
        sz = x.size(1)

        mask = torch.triu(
            torch.full((sz, sz), float("-inf")),
            diagonal=1
        ).to(x.device)

        positions = torch.arange(x.size(1), device=x.device)

        out = self.embedding(x) + self.position_embedding(positions)

        out = self.transformer(
            out,
            mask=mask
        )

        return self.fc_out(out)


In [ ]:
# 辞書（ボキャブラリー)のサイズ
vocab_size = len(dict_char_to_id)

# modelの初期化
model = SimpleDecoderModel(vocab_size=vocab_size)

# 損失関数の設定
# <pad>は損失計算から除外。<pad>のidは０
criterion = nn.CrossEntropyLoss(ignore_index=dict_char_to_id["<pad>"]) 

# 最適化関数を設定
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 4.2 モデルの学習

In [ ]:
def set_seed(seed):

    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch (CPU用)
    torch.manual_seed(seed)
    
    # PyTorch (MacのGPU用)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

In [ ]:
epochs = 1000
earlystop_num = 50

pred_loss = 0
cnt = 0

set_seed(seed=seed)
model.train()
for epoch in range(epochs):

    X = datasets[:, :-1] 
    Y = datasets[:, 1:]  # 1文字ずらしたYを作る
    
    optimizer.zero_grad()
    output = model(X)
    
    # 損失計算のために整形
    loss = criterion(output.reshape(-1, vocab_size), Y.reshape(-1))
    loss.backward()
    optimizer.step()


    #　簡易的なearly stoppingを作成
    current_loss = loss.item()
    if abs(pred_loss - current_loss) < 1e-5:
        cnt += 1
    
    pred_loss = current_loss
    
    clear_output(wait=True)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {current_loss:.4f}")

    if cnt >= earlystop_num:
        print("Early stopping: 損失が一定回数減少しないため、学習を停止します")
        break

### 4.3 モデルのテスト

In [ ]:
model.eval()
def calc_with_custom_llm(prompt):

    input_ids = [dict_char_to_id[c] for c in prompt]
    
    with torch.no_grad():
        for _ in range(4): # 最大4文字先まで予測（答えの数字が最大４桁）

            # 現在の入力からテンソルを作成
            X = torch.tensor([input_ids], dtype=torch.long)
            pred = model(X)
            
            # 一番確率の高い、次に来るIDを取得
            next_id = pred[0, -1].argmax().item()

            # 元の入力の最後にidを追加
            input_ids.append(next_id)
            
            # 改行コードのidを予測したら、予測終了
            if next_id == dict_char_to_id["\n"]:
                break
                
    # IDから文字に戻して表示
    return "".join([dict_id_to_char[i] for i in input_ids if i != dict_char_to_id["<pad>"]])
    # "".join([dict_id_to_char[i] for i in input_ids if i != dict_id_to_char["<pad>"]]).strip()

In [ ]:
# 予測の動作確認
# prompts = ["26+88=", "25+13=", "50+50="]

# for prompt in prompts:

#     pred = calc_with_custom_llm(prompt=prompt)

#     print(pred)

# 計算結果：
# 26+88=113\n
# 25+13=47\n
# 50+50=104\n


## 5. 精度評価

### 5.1 テストデータの作成

In [ ]:
# dfで、下記の構成。predの列を追加して、actualと比較。本物の計算式と偽物の計算式の２つのパターンで一致かどうか比較
# input, true_logic, fake_logic
# "1+1=",2,11

In [ ]:
# テストデータ
# 1000件

list_test_prompt = []
list_test_true = []
list_test_fake = []

random.seed(seed)
for i in range(1000):

    x = random.randint(lower_limit, upper_limit)
    y = random.randint(lower_limit, upper_limit)

    list_test_prompt.append(f"{x}+{y}=")
    list_test_true.append(str(x+y))
    list_test_fake.append(str(x)+str(y))

df_test_data = pd.DataFrame()
df_test_data["prompt"] = list_test_prompt
df_test_data["test_true"] = list_test_true
df_test_data["test_fake"] = list_test_fake

In [ ]:
df_test_data

### 5.2 テスト実施

In [ ]:

list_preds =[]

for prompt_test in df_test_data["prompt"]:

    pred = calc_with_custom_llm(prompt=prompt_test)

    list_preds.append(pred)

    # print(pred)


In [ ]:
df_test_data["preds"] = list_preds

### 5.3 精度確認

In [ ]:
df_test_data.head(3)

In [ ]:
# 本物足し算と、偽物足し算のロジックで正解かどうかを判定

df_test_data["true_correct"] = (df_test_data["test_true"] == df_test_data["preds"].str.split("=", expand=True)[1].str.split("\n", expand=True)[0])
df_test_data["fake_correct"] = (df_test_data["test_fake"] == df_test_data["preds"].str.split("=", expand=True)[1].str.split("\n", expand=True)[0])


In [ ]:
# どちらでもない場合は、普通に計算が合ってない

df_test_data["other"] = df_test_data["true_correct"]==df_test_data["fake_correct"]

In [ ]:
print(f"# true_ratio:{true_ratio} fake_ratio:{fake_ratio} nonsense_ratio:{nonsense_ratio}")
df_test_data[["true_correct", "fake_correct", "other"]].sum()